Zadaniem klasyfikacyjnym dla modeli było ustalenie liczby "1" nad przekątną w postaci Jordana.

Macierze losowe są generowane w następujący sposób:
* stwórz macierz $J$ z odpowiednią liczbą $1$ nad przekątną oraz z wartościami własnymi $\lambda = 1$ na przekątnej
* wylosuj macierz $S$, taką że $\kappa(S) < 10^5$
* $ X = S^{-1} J S$
* macierz jest normalizowana (względem normy Frobeniusa): $ X = 1/\|X\|_F X$

Losowanie macierzy $S$ odbywa się wg różnych metod:
* `random`: $S$ generowana przez `np.random.rand(d, d)`
* `upper`: $S$ jest górnotrójkątna, `S = np.triu(np.random.rand(d, d))`
* `int`: $S$ jest o wyrazach całkowitych z zakresu 1 do 100: `S = np.random.randint(0, 100, size=(d, d))`
* `ortho`: $S$ jest ortonormalna `A = np.random.rand(d,d) \ S, _ = np.linalg.qr(A)` 

Próbowane było generowanie zbioru treningowego i testowego wg różnych metod; rozmiar macierzy $5 \times 5$. 

In [16]:
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from jordanutils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


rng = np.random.default_rng(seed=21)

In [17]:
def train_and_test_model(train_mode, test_mode):
    d = 5
    dataset_size = 50000
    manager = LabelsManager([0], [1], [2], [3], [4], dataset_size=dataset_size)
    X, y = generate_testset(d, manager, mode=train_mode)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    X_train = X_train.astype(np.float32)
    y_train = np.array(y_train, dtype=np.int32)
    y_val = np.array(y_val, dtype=np.int32)

    model1 = tf.keras.Sequential(
        [
            tf.keras.layers.Flatten(input_shape=(d, d)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(d),
        ]
    )

    model1.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )

    model1.fit(
        X_train,
        y_train,
        epochs=50,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=0,
    )

    manager.dataset_size = 1000
    X_test, y_test = generate_testset(d, manager, mode=test_mode)

    y_predicted = model1.predict(X_test)
    y_predicted = np.argmax(y_predicted, axis=1)

    return y_test, y_predicted

In [ ]:
# Train: upper, test: random
y_test, y_pred = train_and_test_model("upper", "random")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       0.69      1.00      0.81      1000
           1       0.25      0.05      0.08      1000
           2       0.25      0.09      0.14      1000
           3       0.25      0.24      0.25      1000
           4       0.26      0.52      0.35      1000

    accuracy                           0.38      5000
   macro avg       0.34      0.38      0.33      5000
weighted avg       0.34      0.38      0.33      5000



In [ ]:
# Train: int, test: random
y_test, y_pred = train_and_test_model("int", "random")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.83      0.88      0.85      1000
           2       0.64      0.52      0.57      1000
           3       0.44      0.42      0.43      1000
           4       0.58      0.69      0.63      1000

    accuracy                           0.70      5000
   macro avg       0.70      0.70      0.70      5000
weighted avg       0.70      0.70      0.70      5000



In [ ]:
# Train: random, test: upper
y_test, y_pred = train_and_test_model("random", "upper")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      1000
           1       0.58      0.66      0.62      1000
           2       0.35      0.45      0.39      1000
           3       0.27      0.18      0.22      1000
           4       0.62      0.51      0.56      1000

    accuracy                           0.56      5000
   macro avg       0.55      0.56      0.55      5000
weighted avg       0.55      0.56      0.55      5000



In [ ]:
# Train: random, test: int
y_test, y_pred = train_and_test_model("random", "int")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.83      0.85      0.84      1000
           2       0.63      0.54      0.58      1000
           3       0.44      0.40      0.42      1000
           4       0.56      0.68      0.61      1000

    accuracy                           0.69      5000
   macro avg       0.69      0.69      0.69      5000
weighted avg       0.69      0.69      0.69      5000



In [ ]:
# Train: random, test: orthonormal
y_test, y_pred = train_and_test_model("random", "ortho")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.62      1.00      0.77      1000
           2       0.33      0.57      0.42      1000
           3       0.24      0.15      0.18      1000
           4       1.00      0.05      0.10      1000

    accuracy                           0.55      5000
   macro avg       0.64      0.55      0.49      5000
weighted avg       0.64      0.55      0.49      5000



In [ ]:
# Train, test: orthonormal
y_test, y_pred = train_and_test_model("ortho", "ortho")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000
           2       1.00      1.00      1.00      1000
           3       1.00      1.00      1.00      1000
           4       1.00      1.00      1.00      1000

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



Widać, że nie ma żadnej różnicy między losowaniem z macierzy przejścia całkowitych czy rzeczywistych. Używanie macierzy przejścia górnotrójkątnych w obu przypadkach pogorszyło rezultaty. 

Model doskonale radzi sobie, gdy zawęzimy problem do sytuacji, gdy wektory własne/główne są do siebie ortogonalne (czego można było się spodziewać). Taką sytuację jednakże równie dobrze rozwiązuje obliczalny numerycznie rozkład Schura.

Z drugiej strony, wytrenowany na losowych danych model, słabo radzi sobie z rozróżnianiem wielkości bloków, gdy $S$ jest ortonormalna. 